A checkpointer remembers one thread.

A Store can remember information across different threads.

In [13]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore
from langchain_core.runnables import RunnableConfig # to avoid "TypeError: manage_preference() missing 1 required positional argument: 'config'""

In [5]:
# State
class State(TypedDict):
    action: str
    preference: str
    result: str

In [14]:
# Node using Store
def manage_preference(state: State, config: RunnableConfig, * , store: BaseStore) -> dict:
    user_id = config['configurable']['user_id']

    namespace = ("preferences", user_id)

    # save preferences
    if state['action'] == 'save':
        store.put(namespace, "units",{
            "value": state['preference']
        })
        return{
            'result': "Preference saved."
        }
    
    # Read preference
    if state['action'] == 'read':
        item = store.get(namespace, "units")

        if item:
            return {"result": item.value['value']}

        return {
            'result': "No preferences found"
        }
    
    return {'result': "unknown action"}



In [15]:
# Build the graph
builder = StateGraph(State)

builder.add_node("manage_preference", manage_preference)

builder.add_edge(START, "manage_preference")
builder.add_edge("manage_preference", END)



In [16]:
# Create Store
store = InMemoryStore()

graph = builder.compile(store=store)

In [32]:
# Save preference
config1 = {
    "configurable": {
        "user_id": "user-42",
        "thread_id": "convo1"
    }
}

In [33]:
result1 = graph.invoke(
    {
        "action": "save",
        "preference": "metric",
        "result": ""
    },
    config1
)

In [34]:
print(result1['result'])

Preference saved.


Read from another thread

In [35]:
config2 = {
    "configurable": {
        "user_id": "user-42",
        "thread_id": "conversation-999"
    }
}

In [36]:
result = graph.invoke(
    {
        "action": "read",
        "preference": "",
        "result": ""
    },
    config2
)

In [37]:
print(result["result"])

metric
